# Stage 4 — Milvus + three indexes (FLAT / HNSW / IVF_FLAT)

Input: `data/chunks/chunks.jsonl` from `3_chunking_embeddings.ipynb` — already has embeddings, so **no Azure OpenAI calls happen in this notebook at all**.

This notebook:

1. Connects to a local Milvus instance (Docker)
2. Defines one shared collection schema
3. Creates **three separate collections** with identical data but different vector indexes — Milvus only allows one index type per vector field per collection, so FLAT/HNSW/IVF_FLAT each need their own collection
4. Inserts all chunks into all three
5. Smoke-tests each collection with one query

Full latency/accuracy benchmarking across the three is Stage 5 — this notebook just gets them built and confirmed working.

**Before running:** start Milvus locally:

```bash
docker compose up -d
```

(from the project root, using the `docker-compose.yml` already in this repo). Give it ~30–60 seconds to become healthy on first start.


In [1]:
import json
import time
from pathlib import Path

from pymilvus import connections, utility, FieldSchema, CollectionSchema, DataType, Collection

In [2]:
MILVUS_HOST = "localhost"
MILVUS_PORT = "19530"

connections.connect(alias="default", host=MILVUS_HOST, port=MILVUS_PORT)

print("connected. existing collections:", utility.list_collections())

connected. existing collections: []


/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_20174/533925877.py:4: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(alias="default", host=MILVUS_HOST, port=MILVUS_PORT)
/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_20174/533925877.py:6: PyMilvusDeprecationWarning: `utility.list_collections` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print("connected. existing collections:", utility.list_collections())


In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHUNKS_PATH = PROJECT_ROOT / "data" / "chunks" / "chunks.jsonl"

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = [json.loads(line) for line in f]

EMBEDDING_DIM = len(all_chunks[0]["embedding"])

print(f"loaded {len(all_chunks)} chunks, embedding dim = {EMBEDDING_DIM}")

loaded 1301 chunks, embedding dim = 1536


## Shared collection schema

One field layout, reused for all three collections. `chunk_id` is the primary key (no Milvus auto-ID — we already have a stable ID from Stage 3). `content` is sized to Milvus's max VARCHAR length (`65535`) to avoid any insert failure on a longer table/page chunk. `image_path` defaults to an empty string for non-image chunks, since Milvus VARCHAR fields don't handle `None` cleanly.


In [4]:
def build_schema() -> CollectionSchema:
    fields = [
        FieldSchema(name="chunk_id", dtype=DataType.VARCHAR, max_length=64, is_primary=True),
        FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=EMBEDDING_DIM),
        FieldSchema(name="doc_id", dtype=DataType.VARCHAR, max_length=256),
        FieldSchema(name="page_number", dtype=DataType.INT64),
        FieldSchema(name="chunk_type", dtype=DataType.VARCHAR, max_length=16),
        FieldSchema(name="content", dtype=DataType.VARCHAR, max_length=65535),
        FieldSchema(name="image_path", dtype=DataType.VARCHAR, max_length=512),
    ]
    return CollectionSchema(fields=fields, description="multi-modal RAG chunks")


def to_records(chunks: list[dict]) -> list[dict]:
    return [
        {
            "chunk_id": c["chunk_id"],
            "embedding": c["embedding"],
            "doc_id": c["doc_id"],
            "page_number": c["page_number"],
            "chunk_type": c["chunk_type"],
            "content": c["content"],
            "image_path": c.get("image_path") or "",
        }
        for c in chunks
    ]


records = to_records(all_chunks)
print(f"prepared {len(records)} records for insertion")

prepared 1301 records for insertion


## Index configurations

Each entry bundles the **build-time** index params with the **search-time** params it needs later — keeping them together here means Stage 5's retriever code can just look up a config by name instead of re-deriving matching parameters.

* **FLAT** — exact brute-force search, no tunable params, slowest but perfectly accurate (this is the accuracy baseline the other two get compared against)
* **HNSW** — graph-based approximate search; `M` controls graph connectivity (higher = more accurate, more memory), `efConstruction` controls build-time search effort; `ef` controls query-time search effort (higher = more accurate, slower)
* **IVF_FLAT** — cluster-based approximate search; `nlist` = number of clusters to build; `nprobe` = how many clusters to search at query time (higher = more accurate, slower)

All three use `COSINE` similarity, matching the cosine distance already used for semantic chunking in Stage 3.


In [5]:
INDEX_CONFIGS = {
    "flat": {
        "index_type": "FLAT",
        "metric_type": "COSINE",
        "index_params": {},
        "search_params": {},
    },
    "hnsw": {
        "index_type": "HNSW",
        "metric_type": "COSINE",
        "index_params": {"M": 16, "efConstruction": 200},
        "search_params": {"ef": 64},
    },
    "ivf_flat": {
        "index_type": "IVF_FLAT",
        "metric_type": "COSINE",
        "index_params": {"nlist": 128},
        "search_params": {"nprobe": 10},
    },
}

## Build, populate, and index one collection

`build_collection` drops any stale collection of the same name first (safe to re-run this notebook), inserts every record, builds the index, then `.load()`s it into memory — Milvus requires an explicit load before a collection is queryable.

Build time is timed and printed per collection, since index build time vs. query time is itself part of the FLAT/HNSW/IVF_FLAT tradeoff you're comparing in Stage 5.


In [6]:
def build_collection(name: str, config: dict, records: list[dict]) -> Collection:
    if utility.has_collection(name):
        utility.drop_collection(name)

    collection = Collection(name=name, schema=build_schema())

    collection.insert(records)
    collection.flush()

    collection.create_index(
        field_name="embedding",
        index_params={
            "index_type": config["index_type"],
            "metric_type": config["metric_type"],
            "params": config["index_params"],
        },
    )
    collection.load()
    return collection

In [7]:
collections = {}

for key, config in INDEX_CONFIGS.items():
    collection_name = f"chunks_{key}"
    start = time.perf_counter()
    collections[key] = build_collection(collection_name, config, records)
    elapsed = time.perf_counter() - start
    print(f"{collection_name}: built + indexed + loaded in {elapsed:.2f}s, rows = {collections[key].num_entities}")

/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_20174/280206251.py:2: PyMilvusDeprecationWarning: `utility.has_collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  if utility.has_collection(name):
/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_20174/280206251.py:5: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection(name=name, schema=build_schema())
/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_20174/280206251.py:7: PyMilvusDeprecationWarning: `Collection.insert` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.insert(records)
/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_20174/280206251.py:8: PyMilvusDeprecationWarning: `Collection.flush` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClie

chunks_flat: built + indexed + loaded in 5.38s, rows = 1301
chunks_hnsw: built + indexed + loaded in 6.34s, rows = 1301
chunks_ivf_flat: built + indexed + loaded in 6.10s, rows = 1301


## Smoke test

Not the real benchmark yet (that's Stage 5, with a proper query set and ground truth) — just confirming each collection actually returns sensible results. Uses an existing chunk's own embedding as the query vector, so the top result should be that same chunk matching itself with a near-perfect similarity score.


In [8]:
sample_chunk = all_chunks[0]
query_vector = sample_chunk["embedding"]

print("query chunk_id:", sample_chunk["chunk_id"])
print("query content preview:", sample_chunk["content"][:100])
print()

for key, collection in collections.items():
    config = INDEX_CONFIGS[key]
    results = collection.search(
        data=[query_vector],
        anns_field="embedding",
        param={"metric_type": config["metric_type"], "params": config["search_params"]},
        limit=3,
        output_fields=["chunk_id", "doc_id", "page_number", "chunk_type"],
    )[0]

    print(f"--- {key} ---")
    for hit in results:
        print(f"  score={hit.distance:.4f} chunk_id={hit.entity.get('chunk_id')} page={hit.entity.get('page_number')}")
    print()

query chunk_id: 94c6b673b4f249d897e4b7a6f2bd2f30
query content preview: 7

The Earth's Energy Budget, Climate Feedbacks and Climate Sensitivity

Coordinating Lead Authors: 



/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_20174/3136939936.py:10: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


--- flat ---
  score=1.0000 chunk_id=94c6b673b4f249d897e4b7a6f2bd2f30 page=1
  score=0.7354 chunk_id=da3ccaf83c3444209cae110859897bd8 page=8
  score=0.7138 chunk_id=4629019fe005444f9f8237fcd4de03c4 page=2

--- hnsw ---
  score=1.0000 chunk_id=94c6b673b4f249d897e4b7a6f2bd2f30 page=1
  score=0.7354 chunk_id=da3ccaf83c3444209cae110859897bd8 page=8
  score=0.7138 chunk_id=4629019fe005444f9f8237fcd4de03c4 page=2

--- ivf_flat ---
  score=1.0000 chunk_id=94c6b673b4f249d897e4b7a6f2bd2f30 page=1
  score=0.7354 chunk_id=da3ccaf83c3444209cae110859897bd8 page=8
  score=0.7138 chunk_id=4629019fe005444f9f8237fcd4de03c4 page=2



/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_20174/3136939936.py:10: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


In [9]:
for key, collection in collections.items():
    n = collection.num_entities
    status = "OK" if n == len(all_chunks) else "MISMATCH"
    print(f"chunks_{key}: {n} rows ({status}, expected {len(all_chunks)})")

chunks_flat: 1301 rows (OK, expected 1301)
chunks_hnsw: 1301 rows (OK, expected 1301)
chunks_ivf_flat: 1301 rows (OK, expected 1301)


/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_20174/3414754907.py:2: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  n = collection.num_entities


## Next: Stage 5 — Retriever pipeline, latency & accuracy benchmarking

Three collections (`chunks_flat`, `chunks_hnsw`, `chunks_ivf_flat`) are built, indexed, loaded, and confirmed working. Stage 5 builds a proper query set with known-relevant chunks, then measures p50/p95 latency and Recall@k across all three — the actual "which index wins" comparison your assignment asks for.
